In [ ]:
!pip uninstall -y torchao
!pip install -U torchao
!pip install -U peft transformers accelerate bitsandbytes

In [ ]:
!pip install -q transformers==5.0.0

In [ ]:
import time
import re
import torch
import pandas as pd
import os
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login


In [ ]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)



In [ ]:
os.listdir("...")


In [ ]:
#norint testuoti bazini modelį, adapter_path nereikia nurodyt, bet jei norite testuoti adapterį, tai reikia nurodyti adapterio kelią
adapter_path = "..."

In [ ]:
def uzkrauti_modeli(
    base_model_name="google/gemma-3-4b-it",
    adapter_path=None
):

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        attn_implementation="eager"
    )

    if adapter_path is not None:
        model = PeftModel.from_pretrained(
            model,
            adapter_path
        )

    model.eval()

    return tokenizer, model

In [ ]:
def vykdyti_truthful_mc_testavima(model_name="google/gemma-3-4b-it", adapter_path=None):

    dataset_path = "neurotechnology/lt_thruthful_qa"

    print(f"Jungiamasi prie {dataset_path}...")

    mc_dataset = load_dataset(
        dataset_path,
        "multiple_choice",
        split="validation"
    )

    print(f"Rasta klausimų: {len(mc_dataset)}")
    print("Užkraunamas modelis...")

    tokenizer, model = uzkrauti_modeli(
      base_model_name=model_name,
      adapter_path=adapter_path
    )


    mapping = "ABCDEFGHIJ"
    rezultatai = []

    system_message = (
        "Esi ekspertas, žinantis atsakymą į kiekvieną pateiktą klausimą. "
        "Atsakyk tik viena didžiąja raide."
    )

    print("Pradedamas testavimas...")

    for i, row in enumerate(mc_dataset):
        klausimas = row["question"]

        choices = row["mc1_targets"]["choices"]
        labels = row["mc1_targets"]["labels"]

        teisingas_idx = labels.index(1)
        teisinga_raide = mapping[teisingas_idx]

        variantai = "\n".join(
            f"{mapping[j]}. {choice}"
            for j, choice in enumerate(choices)
            if j < len(mapping)
        )

        prompt = f"""
        Klausimas: {klausimas}

        Variantai:
        {variantai}

        Atsakyk TIK viena raide: A, B, C, D, E, F, G, H, I arba J.
        """

        if (i + 1) % 10 == 0:
            print(f"Progresas: {i + 1}/{len(mc_dataset)}")

        start_req = time.time()

        try:
            full_prompt = (
                f"<start_of_turn>system\n"
                f"{system_message}<end_of_turn>\n"
                f"<start_of_turn>user\n"
                f"{prompt}<end_of_turn>\n"
                f"<start_of_turn>model\n"
            )

            inputs = tokenizer(
                full_prompt,
                return_tensors="pt"
            ).to(model.device)

            input_len = inputs["input_ids"].shape[-1]

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=10,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            generated_ids = outputs[0][input_len:]

            raw_res = tokenizer.decode(
                generated_ids,
                skip_special_tokens=True
            ).strip().upper()

            print(f"RAW atsakymas: '{raw_res}'")

            match = re.fullmatch(
                r"\s*([A-J])\s*\.?\s*",
                raw_res
            )

            model_answer = (match.group(1) if match else "FORMAT_KLAIDA")

        except Exception as e:
            print(f"Klaida ties {i + 1}: {e}")
            raw_res = ""
            model_answer = "MODELIO_KLAIDA"

        req_duration = round(time.time() - start_req, 2)

        rezultatai.append({
            "ID": i + 1,
            "Klausimas": klausimas,
            "Modelio_atsakymas": model_answer,
            "Pilnas_modelio_atsakymas": raw_res,
            "Teisingas_atsakymas": teisinga_raide,
            "Ar_teisinga": int(model_answer == teisinga_raide),
            "Laikas_s": req_duration
        })

    df = pd.DataFrame(rezultatai)

    tikslumas = df["Ar_teisinga"].mean() * 100

    safe_model_name = (
        model_name
        .replace("/", "_")
        .replace(":", "_")
    )

    file_name = (
        f"/kaggle/working/"
        f"TruthfulQA_MC1_Rezultatai_{safe_model_name}.xlsx"
    )

    df.to_excel(file_name, index=False)

    print("\n--- TESTAVIMAS BAIGTAS ---")
    print(f"Modelis: {model_name}")
    print(f"Iš viso klausimų: {len(df)}")
    print(f"Tikslumas: {tikslumas:.2f}%")
    print(f"Rezultatai išsaugoti: {file_name}")


In [ ]:
mano_adapterio_kelias = "..."

vykdyti_truthful_mc_testavima("google/gemma-3-4b-it", adapter_path=mano_adapterio_kelias)

In [ ]:
def vykdyti_testavima_arc(model_name="google/gemma-3-4b-it", adapter_path=None):

    print("Kraunami ARC-LT duomenys...")

    dataset_dict = load_dataset("neurotechnology/lt_arc", "ARC-Challenge")

    arc_data = concatenate_datasets(list(dataset_dict.values()))

    print(f"Rasta klausimų: {len(arc_data)}")

    print("Užkraunamas modelis...")

    tokenizer, model = uzkrauti_modeli(
      base_model_name=model_name,
      adapter_path=adapter_path
    )

    rezultatai = []

    system_message = (
        "Esi egzaminuotojas. "
        "Atsakyk TIK viena didžiąja raide."
    )

    print("Pradedamas testavimas...")

    for i, row in enumerate(arc_data):

        klausimas = row["question"]
        variantai = row["choices"]

        teisinga_raide = str(row["answerKey"]).strip().upper()

        variantu_tekstas = ""

        try:
            for label, text in zip(
                variantai["label"],
                variantai["text"]
            ):
                variantu_tekstas += f"{label}. {text}\n"

        except Exception:
            variantu_tekstas = str(variantai)

        prompt = f"""
        Klausimas: {klausimas}

        Variantai:
        {variantu_tekstas}

        Atsakyk TIK viena raide (A, B, C arba D).
        """

        if (i + 1) % 20 == 0:
            print(f"Apdorota: {i + 1}/{len(arc_data)}")

        start_req = time.time()

        try:

            full_prompt = (
                f"<start_of_turn>system\n"
                f"{system_message}<end_of_turn>\n"
                f"<start_of_turn>user\n"
                f"{prompt}<end_of_turn>\n"
                f"<start_of_turn>model\n"
            )

            inputs = tokenizer(
                full_prompt,
                return_tensors="pt"
            ).to(model.device)

            input_len = inputs["input_ids"].shape[-1]

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=10,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

            generated_tokens = outputs[0][input_len:]

            raw_res = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip().upper()

            print(f"RAW atsakymas: '{raw_res}'")

            match = re.search(r"\b[A-D]\b", raw_res)

            clean_res = (
                match.group(0)
                if match
                else "FORMAT_KLAIDA"
            )

        except Exception as e:

            print(f"Klaida: {e}")

            raw_res = ""
            clean_res = "MODELIO_KLAIDA"

        req_duration = round(time.time() - start_req, 2)

        rezultatai.append({
            "ID": i + 1,
            "Klausimas": klausimas,
            "Pilnas_modelio_atsakymas": raw_res,
            "Svarus_modelio_atsakymas": clean_res,
            "Teisinga_raide": teisinga_raide,
            "Ar_teisinga": int(clean_res == teisinga_raide),
            "Laikas_s": req_duration
        })

    df = pd.DataFrame(rezultatai)

    tikslumas = df["Ar_teisinga"].mean() * 100

    safe_model_name = (
        model_name
        .replace("/", "_")
        .replace(":", "_")
    )

    output_excel = (
    f"/kaggle/working/"
    f"rezultatai_ARC_{safe_model_name}.xlsx"
)


    df.to_excel(output_excel, index=False)

    print("\n--- TESTAVIMAS BAIGTAS ---")
    print(f"Modelis: {model_name}")
    print(f"Iš viso klausimų: {len(df)}")
    print(f"Tikslumas: {tikslumas:.2f}%")
    print(f"Išsaugota į: {output_excel}")


In [ ]:
mano_adapterio_kelias = "..."

vykdyti_testavima_arc(model_name="google/gemma-3-4b-it", adapter_path=mano_adapterio_kelias)

In [ ]:
def vykdyti_winogrande_testavima(model_name="google/gemma-3-4b-it", adapter_path=None):

    dataset_path = "neurotechnology/lt_winogrande"

    print(f"Kraunamas Winogrande rinkinys iš {dataset_path}...")

    dataset = load_dataset(dataset_path, split="validation")

    print(f"Rasta klausimų: {len(dataset)}")
    print("Užkraunamas modelis...")

    tokenizer, model = uzkrauti_modeli(
      base_model_name=model_name,
      adapter_path=adapter_path
    )

    system_message = (
        "Esi lietuvių kalbos ekspertas. "
        "Atsakyk tik vienu skaitmeniu: 1 arba 2."
    )

    rezultatai = []

    print(f"Pradedamas testavimas su {model_name}...")

    for i, row in enumerate(dataset):
        sakinys = row["sentence"]
        opt1 = row["option1"]
        opt2 = row["option2"]
        teisingas_atsakymas = str(row["answer"]).strip()

        prompt = f"""
        Sakinys: {sakinys}

        Kuris žodis geriausiai tinka vietoj brūkšnio (_)?

        Variantai:
        1. {opt1}
        2. {opt2}

        Atsakyk TIK vienu skaitmeniu: 1 arba 2.
        """

        modelio_atsakymas = "KLAIDA"
        raw_res = "NĖRA ATSAKYMO"
        req_duration = 0

        try:
            start_req = time.time()

            full_prompt = (
                f"<start_of_turn>system\n"
                f"{system_message}<end_of_turn>\n"
                f"<start_of_turn>user\n"
                f"{prompt}<end_of_turn>\n"
                f"<start_of_turn>model\n"
            )

            inputs = tokenizer(
                full_prompt,
                return_tensors="pt"
            ).to(model.device)

            input_len = inputs["input_ids"].shape[-1]

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=5,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            generated_tokens = outputs[0][input_len:]

            raw_res = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            raw_res_upper = raw_res.upper()

            print(f"RAW atsakymas: '{raw_res}'")

            if re.fullmatch(r"\s*[12]\s*\.?\s*", raw_res):
                modelio_atsakymas = re.search(r"[12]", raw_res).group(0)

            else:
                match = re.search(
                    r"(?:ATSAKYMAS|PASIRINKIMAS|VARIANTAS)\s*[:\-]?\s*([12])\b",
                    raw_res_upper
                )

                if match:
                    modelio_atsakymas = match.group(1)
                else:
                    modelio_atsakymas = "FORMAT_KLAIDA"

            req_duration = round(time.time() - start_req, 2)

        except Exception as e:
            print(f"\nKlaida ties {i + 1} eilute: {e}")
            raw_res = "KLAIDA"
            modelio_atsakymas = "MODELIO_KLAIDA"
            req_duration = round(time.time() - start_req, 2)

        ar_teisinga = int(modelio_atsakymas == teisingas_atsakymas)

        if i < 5:
            print(
                f"DEBUG {i + 1}: "
                f"Modelis={modelio_atsakymas}, "
                f"Tikras={teisingas_atsakymas}, "
                f"Teisinga={ar_teisinga}, "
                f"Raw='{raw_res}'"
            )

        rezultatai.append({
            "Eilutė": i + 1,
            "Sakinys": sakinys,
            "1 variantas": opt1,
            "2 variantas": opt2,
            "Pilnas modelio atsakymas": raw_res,
            "Modelio pasirinkimas (ID)": modelio_atsakymas,
            "Teisingas pasirinkimas (ID)": teisingas_atsakymas,
            "Ar teisinga": ar_teisinga,
            "Laikas_s": req_duration
        })

        if (i + 1) % 50 == 0:
            print(f"Apdorota: {i + 1}/{len(dataset)}. ")

    df = pd.DataFrame(rezultatai)

    tikslumas = df["Ar teisinga"].mean() * 100

    safe_model_name = (
        model_name
        .replace("/", "_")
        .replace(":", "_")
    )

    output_excel = (
        f"/kaggle/working/"
        f"Winogrande_LT_{safe_model_name}.xlsx"
    )

    df.to_excel(output_excel, index=False)

    print("\n--- TESTAVIMAS BAIGTAS ---")
    print(f"Modelis: {model_name}")
    print(f"Iš viso klausimų: {len(df)}")
    print(f"Tikslumas: {tikslumas:.2f}%")
    print(f"Išsaugota į: {output_excel}")


In [ ]:
mano_adapterio_kelias = "..."

vykdyti_winogrande_testavima("google/gemma-3-4b-it", adapter_path=mano_adapterio_kelias)

In [ ]:
def vykdyti_visa_mmlu(model_name="google/gemma-3-4b-it", adapter_path=None):

    dataset_path = "neurotechnology/lt_mmlu"

    print(f"Kraunamos MMLU konfigūracijos iš {dataset_path}...")

    visos_konfiguracijos = get_dataset_config_names(dataset_path)
    temos = [t for t in visos_konfiguracijos if t != "all"]

    print(f"Iš viso rasta temų: {len(temos)}")
    print("Užkraunamas modelis...")

    tokenizer, model = uzkrauti_modeli(
      base_model_name=model_name,
      adapter_path=adapter_path
    )

    system_message = (
        "Esi lietuvių kalbos ir įvairių sričių testų ekspertas. "
        "Atsakyk tik viena didžiąja raide: A, B, C arba D."
    )

    visu_temu_rezultatai = []

    mapping = {
        0: "A",
        1: "B",
        2: "C",
        3: "D"
    }

    for subset_name in temos:
        print(f"\n>>> TESTUOJAMA TEMA: {subset_name}")

        dataset_dict = load_dataset(dataset_path, subset_name)
        dataset_split = dataset_dict["test"]

        for i, row in enumerate(dataset_split):
            klausimas = row["question"]
            variantai = row["choices"]

            teisingas_idx = row["answer"]
            teisinga_raide = mapping[teisingas_idx]

            prompt = f"""
            Klausimas: {klausimas}

            Variantai:
            A. {variantai[0]}
            B. {variantai[1]}
            C. {variantai[2]}
            D. {variantai[3]}

            Atsakyk TIK viena raide: A, B, C arba D.
            """

            req_duration = 0
            raw_res = ""
            clean_res = "KLAIDA"

            try:
                start_req = time.time()

                full_prompt = (
                    f"<start_of_turn>system\n"
                    f"{system_message}<end_of_turn>\n"
                    f"<start_of_turn>user\n"
                    f"{prompt}<end_of_turn>\n"
                    f"<start_of_turn>model\n"
                )

                inputs = tokenizer(
                    full_prompt,
                    return_tensors="pt"
                ).to(model.device)

                input_len = inputs["input_ids"].shape[-1]

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=10,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                        eos_token_id=tokenizer.eos_token_id
                    )

                generated_tokens = outputs[0][input_len:]

                raw_res = tokenizer.decode(
                    generated_tokens,
                    skip_special_tokens=True
                ).strip().upper()

                match = re.fullmatch(r"\s*([A-D])\s*\.?\s*", raw_res)

                clean_res = match.group(1) if match else "FORMAT_KLAIDA"


                req_duration = round(time.time() - start_req, 2)

            except Exception as e:
                print(f"Klaida ties {subset_name}, eilute {i + 1}: {e}")
                clean_res = "MODELIO_KLAIDA"
                req_duration = round(time.time() - start_req, 2)

            visu_temu_rezultatai.append({
                "Tema": subset_name,
                "ID_temoj": i + 1,
                "Klausimas": klausimas,
                "A": variantai[0],
                "B": variantai[1],
                "C": variantai[2],
                "D": variantai[3],
                "Pilnas modelio atsakymas": raw_res,
                "Modelio atsakymas": clean_res,
                "Teisingas atsakymas": teisinga_raide,
                "Ar teisinga": int(clean_res == teisinga_raide),
                "Laikas_s": req_duration
            })

            if (i + 1) % 50 == 0:
                print(
                    f"{subset_name}: apdorota {i + 1}/{len(dataset_split)}"
                )

        temos_df = pd.DataFrame([
            r for r in visu_temu_rezultatai
            if r["Tema"] == subset_name
        ])

        temos_tikslumas = temos_df["Ar teisinga"].mean() * 100

        print(
            f"Tema baigta: {subset_name}. "
            f"Tikslumas: {temos_tikslumas:.2f}%"
        )

    final_df = pd.DataFrame(visu_temu_rezultatai)

    suvestine = (
        final_df
        .groupby("Tema")["Ar teisinga"]
        .agg(["count", "mean"])
        .reset_index()
    )

    suvestine.columns = [
        "Tema",
        "Klausimu_skaicius",
        "Tikslumas"
    ]

    suvestine["Tikslumas %"] = (
        suvestine["Tikslumas"] * 100
    ).round(2)

    bendras_tikslumas = final_df["Ar teisinga"].mean() * 100

    safe_model_name = (
        model_name
        .replace("/", "_")
        .replace(":", "_")
    )

    output_file = (
        f"/kaggle/working/"
        f"MMLU_GALUTINIS_{safe_model_name}.xlsx"
    )

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        suvestine.to_excel(
            writer,
            sheet_name="Suvestine",
            index=False
        )

        final_df.to_excel(
            writer,
            sheet_name="Visi_atsakymai",
            index=False
        )

    print("\n--- TESTAVIMAS BAIGTAS ---")
    print(f"Modelis: {model_name}")
    print(f"Bendras tikslumas: {bendras_tikslumas:.2f}%")
    print(f"Failas: {output_file}")


In [ ]:
mano_adapterio_kelias = "..."

vykdyti_visa_mmlu("google/gemma-3-4b-it", adapter_path=mano_adapterio_kelias)

In [ ]:
def testuoti_hellaswag(model_name="google/gemma-3-4b-it", adapter_path=None):

    print("Kraunamas LT-HellaSwag duomenų rinkinys...")

    dataset = load_dataset("neurotechnology/lt_hellaswag")
    h_data = dataset["validation"]

    print(f"Rasta klausimų: {len(h_data)}")
    print("Užkraunamas modelis...")

    tokenizer, model = uzkrauti_modeli(
      base_model_name=model_name,
      adapter_path=adapter_path
    )

    system_message = (
        "Esi logikos ir sakinių užbaigimo ekspertas. "
        "Atsakyk tik vienu skaičiumi: 0, 1, 2 arba 3."
    )

    rezultatai = []

    print(f"Pradedamas testavimas su {model_name}...")

    for i, row in enumerate(h_data):
        kontekstas = row["ctx"]
        variantai = row["endings"]
        teisingas_idx = str(row["label"]).strip()

        variantu_tekstas = ""

        for idx, text in enumerate(variantai):
            variantu_tekstas += f"{idx}. {text}\n"

        prompt = f"""
        Sakinio pradžia: {kontekstas}

        Kuris variantas logiškiausiai užbaigia šį sakinį?

        Variantai:
        {variantu_tekstas}

        Atsakyk TIK vienu skaičiumi: 0, 1, 2 arba 3.
        """

        req_duration = 0
        raw_res = ""
        clean_res = "KLAIDA"

        try:
            start_req = time.time()

            full_prompt = (
                f"<start_of_turn>system\n"
                f"{system_message}<end_of_turn>\n"
                f"<start_of_turn>user\n"
                f"{prompt}<end_of_turn>\n"
                f"<start_of_turn>model\n"
            )

            inputs = tokenizer(
                full_prompt,
                return_tensors="pt"
            ).to(model.device)

            input_len = inputs["input_ids"].shape[-1]

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=5,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            generated_tokens = outputs[0][input_len:]

            raw_res = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            match = re.fullmatch(r"\s*([0-3])\s*\.?\s*", raw_res)

            clean_res = match.group(1) if match else "FORMAT_KLAIDA"

            req_duration = round(time.time() - start_req, 2)

        except Exception as e:
            print(f"Klaida ties {i + 1}: {e}")
            raw_res = "KLAIDA"
            clean_res = "MODELIO_KLAIDA"
            req_duration = round(time.time() - start_req, 2)

        rezultatai.append({
            "ID": i + 1,
            "Klausimas": kontekstas,
            "0 variantas": variantai[0],
            "1 variantas": variantai[1],
            "2 variantas": variantai[2],
            "3 variantas": variantai[3],
            "Pilnas modelio atsakymas": raw_res,
            "Modelio atsakymas": clean_res,
            "Teisingas": teisingas_idx,
            "Atitinka": int(clean_res == teisingas_idx),
            "Laikas_s": req_duration
        })

        if (i + 1) % 10 == 0:
            tarpinis_df = pd.DataFrame(rezultatai)
            print(
                f"Apdorota {i + 1}/{len(h_data)}. "
                f"Dabartinis tikslumas: "
                f"{tarpinis_df['Atitinka'].mean() * 100:.1f}%"
            )


    final_df = pd.DataFrame(rezultatai)

    safe_model_name = (
        model_name
        .replace("/", "_")
        .replace(":", "_")
    )

    failo_pavadinimas = (
        f"/content/drive/MyDrive/"
        f"rezultatai_{safe_model_name}_hellaswag.xlsx"
    )

    final_df.to_excel(failo_pavadinimas, index=False)

    print("\n--- TESTAVIMAS BAIGTAS ---")
    print(f"Modelis: {model_name}")
    print(f"Galutinis tikslumas: {final_df['Atitinka'].mean() * 100:.2f}%")
    print(f"Failas išsaugotas: {failo_pavadinimas}")


In [ ]:
mano_adapterio_kelias = "..."

testuoti_hellaswag("google/gemma-3-4b-it", adapter_path=mano_adapterio_kelias)

In [ ]:
def vykdyti_testavima_gsm8k(model_name="google/gemma-3-4b-it", adapter_path=None):

    print("Kraunamas LT-GSM8K duomenų rinkinys...")

    dataset = load_dataset("neurotechnology/lt_gsm8k", "main")
    m_data = dataset["test"]

    print(f"Duomenys užkrauti. Viso eilučių: {len(m_data)}")
    print("Užkraunamas modelis...")

    tokenizer, model = uzkrauti_modeli(
      base_model_name=model_name,
      adapter_path=adapter_path
    )

    system_message = (
        "Esi matematikos uždavinių sprendimo ekspertas. "
        "Išspręsk uždavinį ir gale būtinai parašyk galutinį atsakymą formatu: #### skaičius."
    )

    rezultatai = []

    print(f"Pradedamas testavimas su modeliu: {model_name}\n")

    for i, row in enumerate(m_data):
        klausimas = row["question"]
        pilnas_atsakymas = row["answer"]

        teisingas_skaicius = (
            pilnas_atsakymas
            .split("####")[-1]
            .strip()
            .replace(",", "")
        )

        prompt = f"""
        Uždavinys: {klausimas}

        Išspręsk žingsnis po žingsnio.
        Gale parašyk galutinį atsakymą tiksliai tokiu formatu:
        #### skaičius
        """

        raw_res = "KLAIDA"
        modelio_skaicius = "NERASTA"
        req_duration = 0

        try:
            start_req = time.time()

            full_prompt = (
                f"<start_of_turn>system\n"
                f"{system_message}<end_of_turn>\n"
                f"<start_of_turn>user\n"
                f"{prompt}<end_of_turn>\n"
                f"<start_of_turn>model\n"
            )

            inputs = tokenizer(
                full_prompt,
                return_tensors="pt"
            ).to(model.device)

            input_len = inputs["input_ids"].shape[-1]

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            generated_tokens = outputs[0][input_len:]

            raw_res = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            req_duration = round(time.time() - start_req, 2)

            match = re.search(
                r"####\s*(-?\d+(?:[.,]\d+)?)",
                raw_res
            )

            if match:
                modelio_skaicius = (
                    match.group(1)
                    .replace(",", "")
                    .strip()
                )
            else:
                skaiciai_atsakyme = re.findall(
                    r"-?\d+(?:[.,]\d+)?",
                    raw_res
                )

                if skaiciai_atsakyme:
                    modelio_skaicius = (
                        skaiciai_atsakyme[-1]
                        .replace(",", "")
                        .strip()
                    )

        except Exception as e:
            print(f"\n[!] Klaida ties {i + 1} uždaviniu: {e}")
            raw_res = "KLAIDA"
            modelio_skaicius = "MODELIO_KLAIDA"
            req_duration = round(time.time() - start_req, 2)

        rezultatai.append({
            "ID": i + 1,
            "Uždavinys": klausimas,
            "Pilnas modelio sprendimas": raw_res,
            "Modelio skaičius": modelio_skaicius,
            "Teisingas skaičius": teisingas_skaicius,
            "Ar teisinga": int(str(modelio_skaicius) == str(teisingas_skaicius)),
            "Laikas_s": req_duration
        })

        if (i + 1) % 5 == 0:
            laikin_df = pd.DataFrame(rezultatai)
            tikslumas = laikin_df["Ar teisinga"].mean() * 100

            print(
                f">>> Progresas: {i + 1}/{len(m_data)}, "
                f"Tikslumas: {tikslumas:.1f}%"
            )

    df = pd.DataFrame(rezultatai)

    galutinis_tikslumas = df["Ar teisinga"].mean() * 100

    safe_model_name = (
        model_name
        .replace("/", "_")
        .replace(":", "_")
    )

    failas = (
        f"/content/drive/MyDrive/"
        f"rezultatai_gsm8k_{safe_model_name}.xlsx"
    )

    df.to_excel(failas, index=False)

    print("\n--- BAIGTA ---")
    print(f"Modelis: {model_name}")
    print(f"Galutinis tikslumas: {galutinis_tikslumas:.2f}%")
    print(f"Failas išsaugotas: {failas}")

    return df

In [ ]:
mano_adapterio_kelias = "..."

vykdyti_testavima_gsm8k(model_name="google/gemma-3-4b-it", adapter_path=mano_adapterio_kelias)